# ML Classifier Experiments

In [1]:
# === ENVIRONMENT & FILEPATH SETUP ===
import os
import sys

codebase_path = "../"
if codebase_path not in sys.path:
    sys.path.insert(0, codebase_path)

# DEBUG: Clear src modules to allow reloading
for key in list(sys.modules.keys()):
    if key.startswith("src"):
        del sys.modules[key]

DATA_DIR = f"{codebase_path}data"
CACHE_DIR = f"{codebase_path}output/cache"
MODELS_DIR = f"{codebase_path}output/models"
ARTIFACTS_DIR = f"{codebase_path}artifacts"

In [2]:
# === IMPORT LIBRARIES ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold
import lightgbm as lgb
from xgboost import XGBClassifier, XGBRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import Ridge

In [3]:
# === IMPORT TRAIN.CSV ===
train = pd.read_csv(f"{DATA_DIR}/train.csv")
models = ['Model_A', 'Model_B', 'Model_C', 'Model_D', 'Model_E', 'Model_F', 'Model_G', 'Model_H', 'Model_I', 'Model_J', 'Model_K']
perf_cols = [f"{m}_performance" for m in models]
cost_cols = [f"{m}_cost" for m in models]
max_cost_per_query = train[cost_cols].max(axis=1)
global_avg_max_cost = max_cost_per_query.mean()

In [4]:
# Create Target: Argmax of custom reward for each row
rewards = pd.DataFrame(index=train.index, columns=models)
for m in models:
    p = train[f"{m}_performance"]
    c = train[f"{m}_cost"]
    rewards[m] = 0.85 * p - 0.15 * (c / global_avg_max_cost)

train['best_model'] = rewards.idxmax(axis=1)
train['best_model_idx'] = train['best_model'].apply(lambda x: models.index(x))

In [ ]:
# === CONFIGURATIONS ===
RUNNING_IN_PIPELINE = False
# Feature Engineering Configuration
FEATURE_CONFIG = "dense_Qwen-Qwen3-Embedding-8B_features"
# MAX_FEATURES = 30000
# NG = "1-2"

# ML Model Configuration
ML_MODEL_TYPE = "lightgbm" # Options: 'lightgbm', 'xgboost', 'random_forest', 'ridge'
ROUTING_ML_MODEL_TYPE = "classification" # Options: 'classification', 'regression', 'cost_sensitive'

# Validation Settings
K_FOLDS = 10
RANDOM_STATE = 42

# Ensure consistency
if ROUTING_ML_MODEL_TYPE == 'regression' and ML_MODEL_TYPE == 'ridge':
    pass # Ridge is valid for regression
elif ROUTING_ML_MODEL_TYPE != 'regression' and ML_MODEL_TYPE == 'ridge':
    raise ValueError("Ridge regression can only be used with ROUTING_METHOD='regression'")

In [ ]:
# === SELECT FEATURE ENGINEERING METHOD ===
import joblib
from scipy import sparse
import numpy as np
import os

if FEATURE_CONFIG.startswith('tfidf'):
    vectorizer_path = f"{CACHE_DIR}/{FEATURE_CONFIG}_vectorizer.joblib"
    features_path = f"{CACHE_DIR}/{FEATURE_CONFIG}_features.npz"

    if os.path.exists(features_path) and os.path.exists(vectorizer_path):
        print(f"Loading cached TF-IDF features ({FEATURE_CONFIG})...")
        try:
            vectorizer = joblib.load(vectorizer_path)
        except Exception as e:
            print(f"Warning: Could not load vectorizer: {e}")
        X_sparse = sparse.load_npz(features_path)
        X = X_sparse.toarray()
else:
    model_path = f"{CACHE_DIR}/{FEATURE_CONFIG}_model.joblib"
    features_path = f"{CACHE_DIR}/{FEATURE_CONFIG}_features.npy"
    
    if os.path.exists(features_path):
        print(f"Loading cached dense features ({FEATURE_CONFIG})...")
        X = np.load(features_path)
        if os.path.exists(model_path):
            try:
                model = joblib.load(model_path)
            except Exception as e:
                print(f"Warning: Could not load model: {e}")

y = train['best_model_idx'].values

In [ ]:
# === K-FOLD CV FOR ROUTING ===

kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=RANDOM_STATE)
oof_preds = np.zeros(len(train))

def get_model(model_type, problem_type):
    if problem_type == 'classification':
        if model_type == 'lightgbm':
            return lgb.LGBMClassifier(n_estimators=100, random_state=42, verbose=-1, n_jobs=4)
        elif model_type == 'xgboost':
            return XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', verbosity=0, n_jobs=4)
        elif model_type == 'random_forest':
            return RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=4)
        elif model_type == 'ridge':
            raise ValueError('Ridge classification is not configured.')
    elif problem_type == 'regression':
        if model_type == 'lightgbm':
            return lgb.LGBMRegressor(n_estimators=100, random_state=42, verbose=-1, n_jobs=4)
        elif model_type == 'xgboost':
            return XGBRegressor(n_estimators=100, random_state=42, verbosity=0, n_jobs=4)
        elif model_type == 'random_forest':
            return RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
        elif model_type == 'ridge':
            return Ridge(alpha=1.0)
    raise ValueError(f'Unknown config: {model_type}, {problem_type}')

if ROUTING_ML_MODEL_TYPE == 'regression':
    print(f"Training 11-way Reward Regressors ({ML_MODEL_TYPE})...")
    y_reg = rewards.values
    oof_pred_rewards = np.zeros((len(train), len(models)))
    
    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X[train_idx], X[val_idx]
        for m_idx in range(len(models)):
            y_train, y_val = y_reg[train_idx, m_idx], y_reg[val_idx, m_idx]
            reg = get_model(ML_MODEL_TYPE, 'regression')
            reg.fit(X_train, y_train)
            oof_pred_rewards[val_idx, m_idx] = reg.predict(X_val)
            
    oof_preds = np.argmax(oof_pred_rewards, axis=1)
elif ROUTING_ML_MODEL_TYPE == 'cost_sensitive':
    print(f"Training Cost-Sensitive Multi-class Classifier ({ML_MODEL_TYPE})...")
    sorted_rewards = np.sort(rewards.values, axis=1)
    sample_weights = sorted_rewards[:, -1] - sorted_rewards[:, -2]
    
    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        w_train = sample_weights[train_idx]
        
        clf = get_model(ML_MODEL_TYPE, 'classification')
        clf.fit(X_train, y_train, sample_weight=w_train)
        oof_preds[val_idx] = clf.predict(X_val)
else:
    print(f"Training Standard Multi-class Classifier ({ML_MODEL_TYPE})...")
    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        clf = get_model(ML_MODEL_TYPE, 'classification')
        clf.fit(X_train, y_train)
        oof_preds[val_idx] = clf.predict(X_val)

train['oof_pred_model'] = [models[int(p)] for p in oof_preds]


In [ ]:
# === CALCULATE CV REWARD 0.85 SCORE ===
pred_p = []
pred_c = []
for i, row in train.iterrows():
    pred_m = row['oof_pred_model']
    pred_p.append(row[f"{pred_m}_performance"])
    pred_c.append(row[f"{pred_m}_cost"])

avg_p = np.mean(pred_p)
avg_c = np.mean(pred_c)
cv_reward = 0.85 * avg_p - 0.15 * (avg_c / global_avg_max_cost)
print(f"CV Reward 0.85:", round(cv_reward, 4))
print("CV Average Performance:", round(avg_p, 4))
print("CV Average Cost:", round(avg_c, 4))

In [ ]:
# === LOG RESULTS INTO EXPERIMENT_SUMMARY.CSV ===
import json
results = {
    'K-Fold': kf.n_splits,
    'CV_Reward_0.85': round(cv_reward, 4),
    'CV_Avg_Performance': round(avg_p, 4),
    'CV_Avg_Cost': round(avg_c, 4),
    'Model_Distribution': str(train['oof_pred_model'].value_counts().to_dict()),
    'Public_Kaggle_Score': None
}

if RUNNING_IN_PIPELINE:
    temp_path = f"{ARTIFACTS_DIR}/results_{FEATURE_CONFIG}_{ML_MODEL_TYPE}_{ROUTING_ML_MODEL_TYPE}.json"
    with open(temp_path, 'w') as f:
        json.dump(results, f)
else:
    summary = pd.read_csv(f"{ARTIFACTS_DIR}/experiment_summary.csv")
    new_row = results.copy()
    summary = pd.concat([summary, pd.DataFrame([new_row])], ignore_index=True)
    summary.to_csv(f"{ARTIFACTS_DIR}/experiment_summary.csv", index=False)
